# Run Real Video VAE Latent Probe

本文件是 `real_video_vae_latent_probe` 运行入口。消费上游 processed dataset handoff 与 session-only VAE 模型, 并把 runner、checker、table rebuild 与结果打包全部委托给仓库模块。

## 00 Runtime Identity And User Config

集中声明本次 probe run 的运行模式、family 身份、processed dataset key、仓库根目录、session local 路径、模型来源与 formal 门禁开关。

`REPO_URL` 与 `REPO_ROOT` 控制 Colab 冷启动时的仓库 bootstrap；
`RUN_MODE` 与 `RUNTIME_PROFILE` 控制运行模式；
`WORKFLOW_KEY`、`STEP_KEY`、`FAMILY_ID` 用于结果包登记；
`PROCESSED_DATASET_KEY` 来自上游 processed dataset notebook 的最终 handoff；
`PROCESSED_DATASET_MANIFEST` 指向 session local 的 manifest；
`MODEL_ID` 指定 session-only 模型来源；
`REQUIRE_FORMAL_PASS=True` 表示正式检查失败时必须阻断打包。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
REPO_URL = os.environ.get('TSTW_REPO_URL', 'https://github.com/RICHAAARC/TSTW-VW.git')
REPO_BRANCH = os.environ.get('TSTW_REPO_BRANCH', 'main')
REPO_ROOT = Path(os.environ.get('TSTW_REPO_ROOT', '/content/TSTW-VW'))
REPO_PACKAGE_MARKER = REPO_ROOT / 'paper_workflow' / '__init__.py'
if not REPO_PACKAGE_MARKER.exists():
    if REPO_ROOT.exists():
        raise RuntimeError(f'Repository root exists but is incomplete: {REPO_ROOT}')
    clone_result = subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            '--branch',
            REPO_BRANCH,
            REPO_URL,
            str(REPO_ROOT),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            'repository bootstrap failed; set TSTW_REPO_URL to an authenticated clone URL or pre-clone the repository into TSTW_REPO_ROOT before running the notebook'
        )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
from paper_workflow.notebook_utils import real_video_vae_latent_probe_workflow as probe_workflow
from paper_workflow.notebook_utils import run_timing_workflow
from paper_workflow.notebook_utils import runtime_profile_workflow
RUN_MODE = 'formal'
RUNTIME_PROFILE = 'formal'
PROFILE_RUNTIME = RUN_MODE == 'formal'
PROFILE_GPU_RUNTIME = PROFILE_RUNTIME
GPU_PROFILE_INTERVAL_SECONDS = 2
PROFILE_DRIVE_IO = PROFILE_RUNTIME
DRIVE_IO_SAMPLE_SIZE_MB = 64
WRITE_RUNTIME_RECOMMENDATION = True
WORKFLOW_KEY = 'real_video_vae_latent_probe_completion_formal'
STEP_KEY = 'step02_run_real_video_vae_latent_probe'
FAMILY_ID_TEMPLATE = 'real_video_vae_latent_probe__formal__davis2017_trainval480p__utc_time__short_commit'
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001'
DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_RUNTIME_ROOT = Path('/content/TSTW_runtime')
LPIPS_MODEL_ROOT = os.environ.get(
    'TSTW_LPIPS_MODEL_ROOT',
    os.environ.get('LPIPS_MODEL_ROOT', ''),
).strip()
REQUIRE_LPIPS_EVIDENCE = os.environ.get(
    'TSTW_REQUIRE_LPIPS_EVIDENCE',
    '1' if RUN_MODE == 'formal' else '0',
) == '1'
ENABLE_LPIPS = bool(LPIPS_MODEL_ROOT)
if REQUIRE_LPIPS_EVIDENCE and not ENABLE_LPIPS:
    raise RuntimeError(
        'formal mechanism evidence requires LPIPS; set TSTW_LPIPS_MODEL_ROOT to a local LPIPS weights/cache directory before running the notebook'
    )
ENABLE_CLIP_SIMILARITY = os.environ.get('TSTW_ENABLE_CLIP_SIMILARITY', '0') == '1'
ENABLE_MOTION_CONSISTENCY = os.environ.get('TSTW_ENABLE_MOTION_CONSISTENCY', '1') == '1'
RUNNER_SAMPLES_PER_ROLE_OVERRIDE_TEXT = os.environ.get('TSTW_SAMPLES_PER_ROLE_OVERRIDE', '').strip()
# 覆盖每个 split-role 的 samples_per_role，用于放大或缩小本轮 runner 的样本覆盖规模。
RUNNER_SAMPLES_PER_ROLE_OVERRIDE = None
if RUNNER_SAMPLES_PER_ROLE_OVERRIDE_TEXT:
    try:
        RUNNER_SAMPLES_PER_ROLE_OVERRIDE = int(RUNNER_SAMPLES_PER_ROLE_OVERRIDE_TEXT)
    except ValueError as exc:
        raise ValueError('TSTW_SAMPLES_PER_ROLE_OVERRIDE must be a positive integer') from exc
    if RUNNER_SAMPLES_PER_ROLE_OVERRIDE < 1:
        raise ValueError('TSTW_SAMPLES_PER_ROLE_OVERRIDE must be a positive integer')
REQUIRE_STAGE2_MECHANISM_PASS = os.environ.get('TSTW_REQUIRE_STAGE2_MECHANISM_PASS', '0') == '1'
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
FAMILY_ID = probe_workflow.materialize_family_id(
    family_id_template=FAMILY_ID_TEMPLATE,
    git_commit=GIT_COMMIT,
)
PROCESSED_DATASET_ROOT = DRIVE_ROOT / 'TSTW' / 'datasets' / 'processed' / PROCESSED_DATASET_KEY
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'datasets' / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = LOCAL_DATASET_ROOT / 'dataset_manifest.json'
LOCAL_MODEL_ROOT = LOCAL_RUNTIME_ROOT / 'session_models' / 'autoencoder_kl'
FAMILY_ROOT = DRIVE_ROOT / 'TSTW' / 'results' / 'families' / FAMILY_ID
RUN_ROOT = LOCAL_RUNTIME_ROOT / 'runs' / 'real_video_vae_latent_probe_formal'
RUN_ID = RUN_ROOT.name
RUNTIME_PROFILE_ROOT = RUN_ROOT / 'runtime_profile'
RUNTIME_CONFIG_PATH = RUN_ROOT / 'artifacts' / 'runtime_config.json'
SESSION_MODEL_MANIFEST_PATH = RUN_ROOT / 'artifacts' / 'session_model_manifest.json'
FORMAL_VALIDATION_SUMMARY_PATH = RUNTIME_PROFILE_ROOT / 'formal_validation_summary.json'
STAGE2_MECHANISM_CONFIG_PATH = Path('configs/protocol/stage2_mechanism_gate.json')
STAGE2_MECHANISM_SUMMARY_PATH = RUN_ROOT / 'artifacts' / 'stage2_mechanism_decision.json'
ATTACK_MATRIX_PATH = Path('configs/attacks/real_video_attack_matrix.json')
ABLATION_CONFIG_PATH = Path('configs/ablation/real_video_vae_latent_ablation.json')
MODEL_ID = 'stabilityai/sd-vae-ft-mse'
REQUIRE_FORMAL_PASS = True
ALLOW_INCONCLUSIVE_REVIEW = os.environ.get('TSTW_ALLOW_INCONCLUSIVE_REVIEW', '0') == '1'
if ALLOW_INCONCLUSIVE_REVIEW:
    # 仅用于审查 INCONCLUSIVE 结果，不代表 formal PASS 或阶段推进。
    REQUIRE_FORMAL_PASS = False
run_timer = run_timing_workflow.start_run_timing(
    run_root=RUN_ROOT,
    run_id=RUN_ID,
)

## 01 Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 02 Read Processed Dataset Registry

确认上游 processed dataset 根目录存在, 并打印当前 `PROCESSED_DATASET_KEY` 与 Drive 侧 processed dataset 路径。

In [ ]:
if not PROCESSED_DATASET_ROOT.exists():
    raise FileNotFoundError(PROCESSED_DATASET_ROOT)
print({
    'processed_dataset_root': str(PROCESSED_DATASET_ROOT),
    'processed_dataset_key': PROCESSED_DATASET_KEY,
})

## 03 Prepare Local Runtime Workspace

复制 processed dataset 到 local runtime, 创建 run root 与 family root, 并解析本次 runner 必须使用的 session local `dataset_manifest.json`。

In [ ]:
runtime_workspace_handoff = probe_workflow.prepare_probe_runtime_workspace(
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    local_dataset_root=LOCAL_DATASET_ROOT,
    family_root=FAMILY_ROOT,
    run_root=RUN_ROOT,
    copy_processed_dataset=True,
)
PROCESSED_DATASET_MANIFEST = Path(runtime_workspace_handoff['local_dataset_manifest_path'])
if not PROCESSED_DATASET_MANIFEST.exists():
    raise FileNotFoundError(PROCESSED_DATASET_MANIFEST)
print(runtime_workspace_handoff)

## 04 Clone Or Update Repository

记录当前仓库 short commit, 设置 `PYTHONPATH`, 并打印 git 工作区状态, 便于结果包追踪到具体代码版本。

In [ ]:
repo_env = dict(os.environ)
repo_env['PYTHONPATH'] = str(REPO_ROOT)
GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'],
    text=True,
    env=repo_env,
    cwd=REPO_ROOT,
).strip()
subprocess.run(['git', 'status', '--short'], check=True, env=repo_env, cwd=REPO_ROOT)

## 05 Install Runtime Dependencies

安装运行所需依赖, 包括测试工具、diffusers 相关依赖、模型加载依赖和视频质量指标依赖。

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        'pytest',
        'diffusers',
        'accelerate',
        'transformers',
        'safetensors',
        'huggingface_hub',
        'lpips',
        'pytorch-msssim',
        'zstandard',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 06 Prepare Session Model

准备 session-only AutoencoderKL 模型目录, 并写出 session model manifest。

In [ ]:
with run_timer.event('model_preparation', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    session_model_manifest = probe_workflow.prepare_probe_session_model(
        model_id=MODEL_ID,
        local_model_root=LOCAL_MODEL_ROOT,
        session_manifest_path=SESSION_MODEL_MANIFEST_PATH,
        revision='main',
    )
    LOCAL_MODEL_ROOT = Path(session_model_manifest['models'][0]['local_path'])
    if not LOCAL_MODEL_ROOT.exists():
        raise FileNotFoundError(LOCAL_MODEL_ROOT)
    assert session_model_manifest['model_policy'] == 'session_only_no_drive_model_storage'

if PROFILE_DRIVE_IO:
    drive_io_profile = runtime_profile_workflow.profile_drive_io(
        run_root=RUN_ROOT,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_RUNTIME_ROOT,
        sample_size_mb=DRIVE_IO_SAMPLE_SIZE_MB,
    )
else:
    drive_io_profile = {'skipped': True}

print(session_model_manifest)
print(drive_io_profile)

## 07 Write Runtime Config

输出 `RUNTIME_CONFIG_PATH` 是后续 runner、runtime manifest 与结果包审计的核心输入。


In [ ]:
runtime_extra_config = {
    'family_id': FAMILY_ID,
    'workflow_key': WORKFLOW_KEY,
    'step_key': STEP_KEY,
    'git_commit': GIT_COMMIT,
    'quality_metrics': {
        'enable_lpips': ENABLE_LPIPS,
        'enable_clip_similarity': ENABLE_CLIP_SIMILARITY,
    },
    'temporal_metrics': {
        'enable_motion_consistency': ENABLE_MOTION_CONSISTENCY,
    },
    'runtime_manifest_overrides': {
        'family_root': str(FAMILY_ROOT),
        'session_model_manifest_path': str(SESSION_MODEL_MANIFEST_PATH),
    },
}
if LPIPS_MODEL_ROOT:
    runtime_extra_config['local_lpips_model_root'] = LPIPS_MODEL_ROOT
runtime_config_handoff = probe_workflow.write_probe_runtime_config(
    runtime_config_path=RUNTIME_CONFIG_PATH,
    execution_environment='colab',
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_dataset_root=LOCAL_DATASET_ROOT,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    vae_model_local_path=LOCAL_MODEL_ROOT,
    dataset_manifest_path=PROCESSED_DATASET_MANIFEST,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    extra_config=runtime_extra_config,
)
print(runtime_config_handoff)
print({
    'require_lpips_evidence': REQUIRE_LPIPS_EVIDENCE,
    'lpips_enabled': ENABLE_LPIPS,
    'lpips_model_root': LPIPS_MODEL_ROOT or None,
    'runner_samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
})
if PROFILE_RUNTIME:
    runtime_environment_snapshot = runtime_profile_workflow.capture_colab_environment(
        run_root=RUN_ROOT,
        run_id=RUN_ID,
        run_mode=RUN_MODE,
        runtime_profile=RUNTIME_PROFILE,
    )
    run_scale_estimate = runtime_profile_workflow.estimate_real_video_vae_latent_run_scale(
        run_root=RUN_ROOT,
        dataset_manifest=PROCESSED_DATASET_MANIFEST,
        attack_matrix=ATTACK_MATRIX_PATH,
        ablation_config=ABLATION_CONFIG_PATH,
        runtime_profile=RUNTIME_PROFILE,
    )
else:
    runtime_environment_snapshot = {'skipped': True}
    run_scale_estimate = {'skipped': True}
print(runtime_environment_snapshot)
print(run_scale_estimate)

## 08 Check GPU And Runtime

作用：运行 runtime preflight, 检查当前运行模式、processed dataset 本地目录和 session model 目录是否满足运行前置条件。

In [ ]:
with run_timer.event('runtime_preflight', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    runtime_check_report = probe_workflow.run_probe_runtime_preflight(
        run_mode=RUN_MODE,
        local_dataset_root=LOCAL_DATASET_ROOT,
        local_model_dirs=[LOCAL_MODEL_ROOT],
    )
print(runtime_check_report)

## 09 Verify Repository Contract

项目合同校验与总审计。


In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], check=True, env=repo_env, cwd=REPO_ROOT)
subprocess.run([sys.executable, 'tools/harness/run_all_audits.py'], check=True, env=repo_env, cwd=REPO_ROOT)

## 10 Run Smoke Tests

执行真实视频 VAE encode/decode smoke 测试, 判断视频读写、VAE metadata 与 digest 稳定性。

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        '-m',
        'smoke',
        'tests/integration/test_real_video_vae_encode_decode_smoke.py',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 11 Run Real Video VAE Latent Formal

启动 probe runner, 由仓库模块生成 records、thresholds、manifests、artifacts 与后续可重建表格所需的证据。

`run_root=RUN_ROOT` 指定 session local 运行目录；`runtime_config_path=RUNTIME_CONFIG_PATH` 提供运行配置；`dataset_manifest=PROCESSED_DATASET_MANIFEST` 显式传入 processed dataset manifest, 防止回退到默认 manifest。

In [ ]:
import importlib
probe_workflow = importlib.reload(probe_workflow)
runtime_profile_workflow = importlib.reload(runtime_profile_workflow)
run_timing_workflow = importlib.reload(run_timing_workflow)
gpu_profile_process = None
runner_error = None
try:
    if PROFILE_GPU_RUNTIME:
        gpu_profile_process = runtime_profile_workflow.start_gpu_runtime_profile(
            run_root=RUN_ROOT,
            interval_seconds=GPU_PROFILE_INTERVAL_SECONDS,
        )
    with run_timer.event('real_video_vae_latent_runner', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
        probe_workflow.run_probe_runner(
            run_root=RUN_ROOT,
            run_mode=RUN_MODE,
            runtime_profile=RUNTIME_PROFILE,
            runtime_config_path=RUNTIME_CONFIG_PATH,
            attack_matrix=ATTACK_MATRIX_PATH,
            ablation_config=ABLATION_CONFIG_PATH,
            dataset_manifest=PROCESSED_DATASET_MANIFEST,
            samples_per_role=RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
            python_executable=sys.executable,
        )
except Exception as error:
    runner_error = error
finally:
    if PROFILE_GPU_RUNTIME:
        runtime_profile_workflow.stop_gpu_runtime_profile(
            gpu_profile_process,
            run_root=RUN_ROOT,
        )
gpu_runtime_summary = runtime_profile_workflow.summarize_gpu_runtime_profile(
    run_root=RUN_ROOT,
)
run_timing_summary = run_timing_workflow.summarize_run_timing(
    run_root=RUN_ROOT,
)
run_progress_snapshot = runtime_profile_workflow.watch_real_video_vae_latent_progress(
    run_root=RUN_ROOT,
)
run_failure_summary = runtime_profile_workflow.summarize_run_failures(
    run_root=RUN_ROOT,
)
if WRITE_RUNTIME_RECOMMENDATION:
    runtime_parameter_recommendation = runtime_profile_workflow.recommend_runtime_parameters(
        run_root=RUN_ROOT,
    )
else:
    runtime_parameter_recommendation = {'skipped': True}
print(gpu_runtime_summary)
print(run_timing_summary)
print(run_progress_snapshot)
print(run_failure_summary)
print(runtime_parameter_recommendation)
if runner_error is not None:
    raise runner_error

## 12 Rebuild Tables And Reports

records 与 manifests 重建 tables、reports、figures 和 failure gallery 等派生产物。

In [ ]:
with run_timer.event('table_and_report_rebuild', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    probe_workflow.rebuild_probe_tables_and_reports(run_root=RUN_ROOT)

## 13 Check Real Video VAE Latent Outputs

校验输出完整性、formal pass 条件、runtime manifest、records-to-artifacts 链路与下一阶段门禁。

In [ ]:
formal_checker_error = None
try:
    with run_timer.event('formal_checker', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
        formal_validation_summary = probe_workflow.check_probe_outputs(
            run_root=RUN_ROOT,
            construction_phase='real_video_vae_latent_probe',
            run_mode=RUN_MODE,
            require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
        )
        RUNTIME_PROFILE_ROOT.mkdir(parents=True, exist_ok=True)
        FORMAL_VALIDATION_SUMMARY_PATH.write_text(
            json.dumps(formal_validation_summary, ensure_ascii=False, indent=2) + '\n',
            encoding='utf-8',
        )
        if not formal_validation_summary['status']:
            raise RuntimeError(formal_validation_summary)
except RuntimeError as error:
    formal_checker_error = error

run_timing_summary = run_timing_workflow.summarize_run_timing(
    run_root=RUN_ROOT,
)

run_failure_summary = runtime_profile_workflow.summarize_run_failures(
    run_root=RUN_ROOT,
)
if WRITE_RUNTIME_RECOMMENDATION:
    runtime_parameter_recommendation = runtime_profile_workflow.recommend_runtime_parameters(
        run_root=RUN_ROOT,
    )
else:
    runtime_parameter_recommendation = {'skipped': True}

print(formal_validation_summary)
print(run_timing_summary)
print(run_failure_summary)
print(runtime_parameter_recommendation)
if formal_checker_error is not None:
    raise formal_checker_error

## 14 Run Stage2 Mechanism Audit

基于既有 records、thresholds 与 tables 重建阶段 2 mechanism audit, 区分 implementation completion 与 mechanism evidence。

`mechanism_config_path=STAGE2_MECHANISM_CONFIG_PATH` 控制门禁阈值；
`stage2_mechanism_summary` 会写回 run_root 并继续传给 family packaging。

In [ ]:
with run_timer.event('stage2_mechanism_audit', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    stage2_mechanism_summary = probe_workflow.run_probe_stage2_mechanism_audit(
        run_root=RUN_ROOT,
        mechanism_config_path=STAGE2_MECHANISM_CONFIG_PATH,
    )
event_scores_path = RUN_ROOT / 'records' / 'event_scores.jsonl'
if not event_scores_path.exists():
    raise FileNotFoundError(event_scores_path)
lpips_evidence_summary = {
    'lpips_required': REQUIRE_LPIPS_EVIDENCE,
    'lpips_model_root': LPIPS_MODEL_ROOT or None,
    'lpips_record_count': 0,
    'lpips_failure_count': 0,
    'samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
}
with event_scores_path.open('r', encoding='utf-8') as handle:
    for line in handle:
        record = json.loads(line)
        quality_metrics = record.get('quality_metrics', {})
        if not isinstance(quality_metrics, dict):
            continue
        if quality_metrics.get('watermarked_video_lpips') is not None:
            lpips_evidence_summary['lpips_record_count'] += 1
        lpips_failure_reason = quality_metrics.get('lpips_failure_reason')
        if lpips_failure_reason not in {None, 'lpips_disabled_by_config'}:
            lpips_evidence_summary['lpips_failure_count'] += 1
print(stage2_mechanism_summary)
print(lpips_evidence_summary)
print({
    'stage2_mechanism_summary_path': str(STAGE2_MECHANISM_SUMMARY_PATH),
    'require_stage2_mechanism_pass': REQUIRE_STAGE2_MECHANISM_PASS,
})
if REQUIRE_LPIPS_EVIDENCE and lpips_evidence_summary['lpips_record_count'] == 0:
    raise RuntimeError(lpips_evidence_summary)
if REQUIRE_STAGE2_MECHANISM_PASS and stage2_mechanism_summary['Stage2MechanismDecision'] != 'PASS':
    raise RuntimeError(stage2_mechanism_summary)

## 15 Package Family Results

生成 family 结果包、登记 registry, 并打印最终摘要。

`formal_validation_summary` 作为 implementation gate 证据；
`stage2_mechanism_summary` 作为 mechanism gate 证据并写入 family summary。

In [ ]:
with run_timer.event('result_packaging', run_mode=RUN_MODE, runtime_profile=RUNTIME_PROFILE):
    package_payload = probe_workflow.package_probe_family_results(
        run_root=RUN_ROOT,
        family_root=FAMILY_ROOT,
        require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
        formal_validation_summary=formal_validation_summary,
        drive_root=DRIVE_ROOT,
        family_id=FAMILY_ID,
        workflow_key=WORKFLOW_KEY,
        step_key=STEP_KEY,
        run_mode=RUN_MODE,
        mechanism_summary=stage2_mechanism_summary,
    )
drive_archive_path = package_payload['drive_archive_path']
compat_pack_root = package_payload['compat_pack_root']
run_timing_summary = run_timing_workflow.summarize_run_timing(
    run_root=RUN_ROOT,
)
final_family_summary = {
    'family_id': FAMILY_ID,
    'run_root': str(RUN_ROOT),
    'family_root': str(FAMILY_ROOT),
    'runtime_environment_snapshot': runtime_environment_snapshot,
    'drive_io_profile': drive_io_profile,
    'run_scale_estimate': run_scale_estimate,
    'gpu_runtime_summary': gpu_runtime_summary,
    'run_timing_summary': run_timing_summary,
    'run_progress_snapshot': run_progress_snapshot,
    'run_failure_summary': run_failure_summary,
    'runtime_parameter_recommendation': runtime_parameter_recommendation,
    'formal_validation_summary': formal_validation_summary,
    'stage2_mechanism_summary': stage2_mechanism_summary,
    'lpips_evidence_summary': lpips_evidence_summary,
    'require_lpips_evidence': REQUIRE_LPIPS_EVIDENCE,
    'runner_samples_per_role_override': RUNNER_SAMPLES_PER_ROLE_OVERRIDE,
    'drive_archive_path': str(drive_archive_path),
    'compat_pack_root': str(compat_pack_root),
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
}
print({
    'drive_archive_path': drive_archive_path,
    'compat_pack_root': compat_pack_root,
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
})
print(run_timing_summary)
print(final_family_summary)